In [0]:
%run "../00_common/01_environment_config"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Constructors Dimension
# MAGIC
# MAGIC 1. Read Silver `constructors`
# MAGIC 2. Read Gold `ref_nationality_region`
# MAGIC 3. Join using `nationality`
# MAGIC 4. Select required columns
# MAGIC 5. Write to Gold `dim_constructors`


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

constructors_path = f"{s3_root_path}/Silver/Kalashri/constructors"

ref_path = f"{s3_root_path}/Gold/Kalashri/ref_nationality_region"

target_path = f"{s3_root_path}/Gold/Kalashri/dim_constructors"

print("Constructors Path :", constructors_path)
print("Reference Path    :", ref_path)
print("Target Path       :", target_path)


# COMMAND ----------

# ============================================================
# STEP 1 - READ SILVER CONSTRUCTORS
# ============================================================

constructors_df = (
    spark.read
        .format("delta")
        .load(constructors_path)
)

display(constructors_df.limit(20))


# COMMAND ----------

constructors_df.printSchema()


# COMMAND ----------

# ============================================================
# STEP 2 - READ GOLD NATIONALITY-REGION REFERENCE
# ============================================================

ref_nationality_region_df = (
    spark.read
        .format("delta")
        .load(ref_path)
)

display(ref_nationality_region_df.limit(20))


# COMMAND ----------

ref_nationality_region_df.printSchema()


# COMMAND ----------

# ============================================================
# STEP 3 - JOIN CONSTRUCTORS WITH NATIONALITY REGION
# ============================================================
# LEFT JOIN keeps all constructors.
# If nationality does not match the reference data,
# nationality_region will be NULL.
# ============================================================

dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region_df,
            constructors_df.nationality ==
            ref_nationality_region_df.nationality,
            "left"
        )
        .select(
            constructors_df.constructor_id,
            constructors_df.name.alias("constructor_name"),
            constructors_df.nationality,
            ref_nationality_region_df.region.alias(
                "nationality_region"
            )
        )
)

display(dim_constructors_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 4 - CHECK RECORD COUNTS
# ============================================================

print("Silver Constructor Records :", constructors_df.count())
print("Gold Dimension Records     :", dim_constructors_df.count())


# COMMAND ----------

# ============================================================
# STEP 5 - WRITE TO GOLD
# ============================================================

(
    dim_constructors_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target_path)
)


# COMMAND ----------

# ============================================================
# STEP 6 - VALIDATE GOLD DATA
# ============================================================

gold_dim_constructors_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_dim_constructors_df.limit(20))


# COMMAND ----------

gold_dim_constructors_df.printSchema()


# COMMAND ----------

print(
    "Silver Constructor Records :",
    constructors_df.count()
)

print(
    "Gold Constructor Records   :",
    gold_dim_constructors_df.count()
)

print(
    "Gold dim_constructors successfully written to:"
)

print(target_path)